# Colab Training Notebook for spiking-transformer-cs4782
# This notebook sets up a Colab GPU runtime, prepares enwik8, and runs training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/DeepLearning/spikegpt/spiking-transformer-cs4782 /content/
%cd /content/spiking-transformer-cs4782

In [ ]:

!pip install -r requirements.txt
!pip install spikingjelly


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
!wget http://mattmahoney.net/dc/enwik8.zip
!unzip -o enwik8.zip


In [ ]:
from pathlib import Path

raw = Path("enwik8").read_bytes()

out_dir = Path("data/enwik8_split")
out_dir.mkdir(parents=True, exist_ok=True)

n = len(raw)
train_end = int(0.9 * n)
valid_end = int(0.95 * n)

(out_dir / "train.txt").write_bytes(raw[:train_end])
(out_dir / "valid.txt").write_bytes(raw[train_end:valid_end])
(out_dir / "test.txt").write_bytes(raw[valid_end:])

print("done")
print("train:", (out_dir / "train.txt").stat().st_size)
print("valid:", (out_dir / "valid.txt").stat().st_size)
print("test:", (out_dir / "test.txt").stat().st_size)


In [ ]:
# cell to update all the code if you change it
import sys
import importlib
from pathlib import Path

code_path = str(Path("code").resolve())
if code_path not in sys.path:
    sys.path.insert(0, code_path)

for name in ["model", "config", "dataset", "train", "test", "generate"]:
    if name in sys.modules:
        importlib.reload(sys.modules[name])

import model
import config
import dataset
import train
import test
import generate

from model import SpikingGPT
from config import GPTConfig, TrainerConfig
from dataset import Enwik8Dataset

print("project modules refreshed")



In [ ]:
# test before actual training 
import sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path("code").resolve()))

from config import GPTConfig
from model import SpikingGPT
from dataset import Enwik8Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

cfg = GPTConfig(ctx_len=64, n_embd=64, n_layer=2)

dataset = Enwik8Dataset("data/enwik8_split/train.txt", cfg.ctx_len)
x, y = dataset[0]
x = x.unsqueeze(0).to(device)
y = y.unsqueeze(0).to(device)

model = SpikingGPT(cfg).to(device)

print("x shape:", x.shape)
print("y shape:", y.shape)

loss = model(x, y)
print("smoke test loss:", loss.item())


In [ ]:
!ls data/enwik8_split
!python code/train.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!mkdir -p /content/drive/MyDrive/spiking-transformer-cs4782-results
!cp -r results /content/drive/MyDrive/spiking-transformer-cs4782-results/


In [ ]:
!python code/test.py
